# TACO6 YOLOv6 Nano Training

This notebook prepares the Kaggle TACO Trash Dataset, runs a 1-epoch smoke test, trains YOLOv6 nano at 512x288, saves sample inference images, and exports ONNX. It assumes the Kaggle dataset `kneroma/tacotrashdataset` is attached and this repository is available through `PROJECT_ROOT`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import urllib.request

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/kaggle/working/GDG-hackathon-2026"))
TACO_SOURCE = Path(os.environ.get("TACO_SOURCE", "/kaggle/input/tacotrashdataset"))
YOLOV6_DIR = Path(os.environ.get("YOLOV6_DIR", "/kaggle/working/YOLOv6"))
SMOKE_OUT = Path(os.environ.get("SMOKE_OUT", "/kaggle/working/taco6_yolov6_smoke"))
DATASET_OUT = Path(os.environ.get("DATASET_OUT", "/kaggle/working/taco6_yolov6"))
RUNS_OUT = Path(os.environ.get("RUNS_OUT", "/kaggle/working/yolov6_runs"))

converter = PROJECT_ROOT / "training" / "prepare_taco_yolov6.py"
assert converter.exists(), f"Set PROJECT_ROOT to this repo. Missing: {converter}"
assert TACO_SOURCE.exists(), f"Attach kneroma/tacotrashdataset or set TACO_SOURCE. Missing: {TACO_SOURCE}"

def run(cmd, cwd=None):
    cmd = [str(part) for part in cmd]
    print("$", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


## Prepare a Small Smoke Dataset

In [ ]:
run([
    sys.executable,
    converter,
    "--source-root", TACO_SOURCE,
    "--output-dir", SMOKE_OUT,
    "--limit-images", "120",
    "--seed", "2026",
    "--overwrite",
])


## Install YOLOv6 and Patch Training Seed

In [ ]:
if not YOLOV6_DIR.exists():
    run(["git", "clone", "--depth", "1", "https://github.com/meituan/YOLOv6.git", YOLOV6_DIR])

run([sys.executable, "-m", "pip", "install", "-q", "-r", YOLOV6_DIR / "requirements.txt"])
run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxsim"])

weights_dir = YOLOV6_DIR / "weights"
weights_dir.mkdir(exist_ok=True)
weights_path = weights_dir / "yolov6n.pt"
if not weights_path.exists():
    urllib.request.urlretrieve(
        "https://github.com/meituan/YOLOv6/releases/download/0.4.0/yolov6n.pt",
        weights_path,
    )

train_py = YOLOV6_DIR / "tools" / "train.py"
train_text = train_py.read_text()
old_seed_line = "set_random_seed(1+args.rank, deterministic=(args.rank == -1))"
new_seed_line = "set_random_seed(2026 + max(args.rank, 0), deterministic=(args.rank == -1))"
if old_seed_line in train_text:
    train_py.write_text(train_text.replace(old_seed_line, new_seed_line))
elif new_seed_line not in train_text:
    raise RuntimeError("YOLOv6 train.py seed line changed; patch it manually before training.")


## One-Epoch Smoke Train

In [ ]:
run([
    sys.executable, "tools/train.py",
    "--data-path", SMOKE_OUT / "dataset.yaml",
    "--conf-file", "configs/yolov6n_finetune.py",
    "--output-dir", RUNS_OUT,
    "--name", "taco6_smoke_yolov6n_512x288",
    "--epochs", "1",
    "--batch-size", "16",
    "--eval-final-only",
    "--specific-shape",
    "--height", "288",
    "--width", "512",
    "--fuse_ab",
    "--device", "0",
], cwd=YOLOV6_DIR)


## Prepare Full TACO6 Dataset

In [ ]:
run([
    sys.executable,
    converter,
    "--source-root", TACO_SOURCE,
    "--output-dir", DATASET_OUT,
    "--seed", "2026",
    "--overwrite",
])


## Full Balanced Training Run

In [ ]:
run([
    sys.executable, "tools/train.py",
    "--data-path", DATASET_OUT / "dataset.yaml",
    "--conf-file", "configs/yolov6n_finetune.py",
    "--output-dir", RUNS_OUT,
    "--name", "taco6_yolov6n_512x288",
    "--epochs", "150",
    "--batch-size", "32",
    "--eval-interval", "10",
    "--specific-shape",
    "--height", "288",
    "--width", "512",
    "--fuse_ab",
    "--device", "0",
], cwd=YOLOV6_DIR)


## Sample Inference and ONNX Export

In [ ]:
runs = sorted(RUNS_OUT.glob("taco6_yolov6n_512x288*"), key=lambda path: path.stat().st_mtime)
assert runs, "No full training run found."
run_dir = runs[-1]
best_weights = run_dir / "weights" / "best_ckpt.pt"
assert best_weights.exists(), f"Missing best weights: {best_weights}"

run([
    sys.executable, "tools/infer.py",
    "--weights", best_weights,
    "--source", DATASET_OUT / "images" / "val",
    "--yaml", DATASET_OUT / "dataset.yaml",
    "--img-size", "288", "512",
    "--conf-thres", "0.25",
    "--project", RUNS_OUT / "inference",
    "--name", "taco6_samples",
    "--device", "0",
], cwd=YOLOV6_DIR)

run([
    sys.executable, "deploy/ONNX/export_onnx.py",
    "--weights", best_weights,
    "--img-size", "288", "512",
    "--batch-size", "1",
    "--simplify",
    "--device", "0",
], cwd=YOLOV6_DIR)

print("Best weights:", best_weights)
print("ONNX:", best_weights.with_suffix(".onnx"))
print("Dataset summary:", DATASET_OUT / "summary.json")
